# stepa330: проверка очистки обучающих целей

Контроль использует признаки и CatBoost из stepa330. Меняется только вес редких очень больших обучающих целей: 1, 0.25 или 0. Порог оценивается отдельно для маршрута и получасового слота только на разрешённой обучающей истории. Это гипотеза о загрязнении, а не установленные ошибки данных. Нули, исходная история для лагов/профиля и факты tuning/test сохраняются. Этот опыт не проверяет очистку входных признаков.

Train: до 4 октября; tuning: 4–17 октября; test: 18–31 октября 2025. Даты определяются из истории так же, как в stepa330. Октябрьский test уже просматривался в проекте и не является новым слепым периодом. Вариант выбирается только по tuning; независимая от этого выбора оценка test получается до финального переобучения. Одинаковая постобработка выбирается на контроле и затем фиксируется для всех вариантов, чтобы измерить эффект весов.

Импортируйте ноутбук в Kaggle, подключите train/test parquet, включите GPU и запустите все ячейки. Обучение выполняет пользователь. Это новый прогон контроля, численное совпадение с прежним GPU-запуском не гарантируется.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import json
import gc
DATA = Path('/kaggle/input/datasets/stepzak/rwb-ds')
TRAIN_FILE = 'train_solo_track.parquet'
TEST_FILE = 'test_solo_track.parquet'
if not (DATA / TRAIN_FILE).is_file() and Path('/kaggle/input').exists():
    matches = [p.parent for p in Path('/kaggle/input').rglob(TRAIN_FILE) if (p.parent / TEST_FILE).is_file()]
    if len(matches) != 1:
        raise ValueError(f'Укажите DATA вручную. Найденные папки: {matches}')
    DATA = matches[0]
OUT = Path('artifacts/stepa_cleaning_v1/features')
OUT.mkdir(parents=True, exist_ok=True)
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
STEP = pd.Timedelta(minutes=30)
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
PERMUTATION_ENABLED = True
PERMUTATION_TOP_N = 10
PERMUTATION_REPEATS = 3
PERMUTATION_MAX_DAYS = 7
FEATURE_PIPELINE_VERSION = 'team_scaled_experiment_v1'
PILOT_H6_FEATURES = ['start_sin', 'start_cos', 'peak7_mean_max', 'peak7_mean_level', 'peak7_share_-1', 'peak7_share_0', 'peak7_share_1']
CLEANING_WEIGHTS = {'baseline': 1.0, 'soft_spikes': 0.25, 'drop_spikes': 0.0}


In [ ]:
history = pd.read_parquet(DATA / TRAIN_FILE, columns=['route_id', 'timestamp', TARGET, *STATUSES]).sort_values(['timestamp', 'route_id'])
test = pd.read_parquet(DATA / TEST_FILE, columns=['id', 'route_id', 'timestamp'])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if not (history.notna().all().all() and test.notna().all().all()):
    raise ValueError('В данных есть пропуски')
if not not history.duplicated(['route_id', 'timestamp']).any():
    raise ValueError('Повторяется ключ маршрута и времени')
if not times.equals(pd.date_range(times[0], times[-1], freq=STEP)):
    raise ValueError('Нарушена получасовая сетка')
if not len(history) == len(times) * len(routes):
    raise ValueError('Неполная панель маршрутов')
if not (test.id.is_unique and (not test.duplicated(['route_id', 'timestamp']).any())):
    raise ValueError('Повторы идентификаторов или ключей теста')
end = times[-1]
if not (end.hour, end.minute) == (10, 30):
    raise ValueError('Последняя точка истории должна быть в 10:30')
expected = pd.MultiIndex.from_product([routes, [end + h * STEP for h in range(1, 9)]], names=['route_id', 'timestamp'])
if not expected.difference(pd.MultiIndex.from_frame(test[['route_id', 'timestamp']])).empty:
    raise ValueError('Тест не покрывает все ожидаемые маршруты и горизонты')
if not len(test) == len(expected):
    raise ValueError('Число строк теста отличается от ожидаемого')
values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes)) for name in [TARGET, *STATUSES]}
del history
if not all((np.isfinite(array).all() for array in values.values())):
    raise ValueError('В числовых рядах есть бесконечные значения')
dates = times[(times.hour == 10) & (times.minute == 30) & (times + 8 * STEP <= end)]
if len(dates) <= 28:
    raise ValueError('Недостаточно дат для временного разбиения')
tune_start, holdout_start = (dates[-28], dates[-14])
LAG_VALIDATION_FIT_BEFORE = tune_start.normalize()


## Контекстный порог

Для каждой пары маршрут/слот порог равен максимуму из 8 медиан, медианы + 12 × 1.4826 × MAD и 4 общих медиан маршрута. Требуется минимум 28 наблюдений пары. Все статистики обучаются до tuning; при финальном refit — на всей доступной размеченной истории с тем же правилом. Порог намеренно консервативный, константы в этом опыте не перебираются.


In [ ]:
def fit_spike_limits(frame):
    work = frame[['route_id', 'timestamp', 'target_1h']].copy()
    work['slot'] = work.timestamp.dt.hour * 2 + work.timestamp.dt.minute // 30
    keys = ['route_id', 'slot']
    grouped = work.groupby(keys).target_1h
    work['median'] = grouped.transform('median')
    work['deviation'] = (work.target_1h - work['median']).abs()
    limits = work.groupby(keys).agg(median=('target_1h', 'median'), count=('target_1h', 'size'), mad=('deviation', 'median')).reset_index()
    typical = work.groupby('route_id').target_1h.median()
    limits['limit'] = np.maximum.reduce([8 * limits['median'], limits['median'] + 12 * 1.4826 * limits['mad'], 4 * limits.route_id.map(typical)])
    limits.loc[limits['count'] < 28, 'limit'] = np.inf
    return limits

def spike_mask(frame, limits, target_column='y'):
    lookup = limits.set_index(['route_id', 'slot'])['limit']
    slot = frame.timestamp.dt.hour * 2 + frame.timestamp.dt.minute // 30
    index = pd.MultiIndex.from_arrays([frame.route_id, slot])
    bound = lookup.reindex(index).to_numpy()
    return frame[target_column].to_numpy() > bound
raw_labels = pd.read_parquet(DATA / TRAIN_FILE, columns=['route_id', 'timestamp', TARGET])
label_train = raw_labels.loc[raw_labels.timestamp < tune_start.normalize()].copy()
validation_limits = fit_spike_limits(label_train)
validation_limits.to_csv(OUT.parent / 'spike_limits_train.csv', index=False)
flags = spike_mask(label_train, validation_limits, TARGET)
label_train.loc[flags].to_csv(OUT.parent / 'suspicious_train_labels.csv', index=False)
display(pd.DataFrame([{'train_rows': len(label_train), 'flagged': int(flags.sum()), 'row_share': float(flags.mean()), 'target_volume_share': float(label_train.loc[flags, TARGET].sum() / label_train[TARGET].sum())}]))
del label_train, flags


## Исходные признаки stepa330

Лаги и сезонный профиль строятся из неизменённых наблюдений. Позиции временной сетки сохраняются.


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
target = values[TARGET]

def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)
means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
level24 = rolling(target, 48)
positions = np.arange(len(times))
extra_signals = {}
for name in STATUSES[:3]:
    extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)
extra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))
extra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))
for name in STATUSES[3:]:
    extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)
extra_signals['hub_buffer'] = values['status_5'] - values['status_4']
extra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']
extra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)
hub_signal = extra_signals['status_6_diff_hourly']

def common_features(origins):
    common = {}
    for k in (0, 1, 2, 4):
        common[f'target_back_{k}'] = past(target, origins - k)
    for name, array in values.items():
        for steps in (4, 12):
            common[f'{name}_mean_{steps}'] = means[name, steps][origins]
        if name in STATUSES:
            common[f'{name}_latest'] = array[origins]
            if name in STATUSES[:3]:
                common[f'{name}_change_1h'] = array[origins] - past(array, origins - 2)
    for name, array in extra_signals.items():
        common[name] = array[origins]
    for lag in (48, 96):
        common[f'target_1h_lag_{lag}'] = past(target, origins - lag)
    common['warehouse_total_load_lag_48'] = past(extra_signals['warehouse_total_load'], origins - 48)
    common['status_6_diff_hourly_lag_48'] = past(hub_signal, origins - 48)
    common['target_vs_mean_6h'] = target[origins] - means[TARGET, 12][origins]
    recent = means[TARGET, 4][origins]
    usual = average_available([past(means[TARGET, 4], origins - day * 48) for day in range(1, 8)])
    common['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)
    return common
CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']
FEATURE_GROUPS = {'calendar': ['hour', 'minute', 'hour_float', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend', 'weekday_sin', 'weekday_cos'], 'local_statuses': ['status_1_hourly', 'status_2_hourly', 'status_3_hourly', 'warehouse_total_load', 'warehouse_processing_rate'], 'hub_combinations': ['status_4_diff_hourly', 'status_5_diff_hourly', 'status_6_diff_hourly', 'hub_buffer', 'hub_gate_bottleneck'], 'historical_lags': ['target_1h_lag_48', 'target_1h_lag_96', 'warehouse_total_load_lag_48', 'status_6_diff_hourly_lag_48'], 'target_change': ['target_acceleration'], 'correlation_lag': CORRELATION_COLUMNS, 'pilot_start_time': ['start_sin', 'start_cos'], 'pilot_peaks': ['peak7_mean_max', 'peak7_mean_level', 'peak7_share_-1', 'peak7_share_0', 'peak7_share_1']}


In [ ]:
def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError('Идентификаторы маршрутов повторяются')
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError('min_pairs должен быть целым числом >=3')
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError('Некорректные параметры выбора корреляции')
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
            divisor = np.maximum(count, 1)
            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

def add_correlation_features(frame, h, lag_table, chunk_rows=200000):
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError('Некорректный горизонт или размер блока')
    by_route = lag_table.set_index('route_id').reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    quality = {name: by_route[name].to_numpy(dtype=np.float32) for name in CORRELATION_COLUMNS[:4]}
    for name in CORRELATION_COLUMNS:
        frame[name] = np.full(len(frame), np.nan, dtype=np.float32)
    column_positions = {name: frame.columns.get_loc(name) for name in CORRELATION_COLUMNS}
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError('Строка признаков не сопоставлена с историей')
        selected = lag[route]
        source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64)
        available = np.isfinite(selected) & (selected >= h) & (source >= 0) & (source <= origin) & (source < len(times))
        signal = hub_signal[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
        available &= np.isfinite(signal)
        for name, array in quality.items():
            frame.iloc[start:finish, column_positions[name]] = array[route]
        frame.iloc[start:finish, column_positions['lagged_status6_signal']] = np.where(available, signal, np.nan)
        frame.iloc[start:finish, column_positions['lagged_status6_signal_available']] = available.astype(np.float32)
    return frame
validation_lag_table = calculate_dynamic_shoulder(target, hub_signal, times, routes, fit_before=LAG_VALIDATION_FIT_BEFORE, max_lookback_hours=LAG_MAX_LOOKBACK_HOURS, min_pairs=LAG_MIN_PAIRS, min_correlation=LAG_MIN_CORRELATION)
validation_lag_table.to_csv(OUT.parent / 'status6_lags_validation.csv', index=False)
(OUT.parent / 'feature_pipeline.json').write_text(json.dumps({'version': FEATURE_PIPELINE_VERSION, 'validation_lag_fit_before': str(LAG_VALIDATION_FIT_BEFORE), 'max_lookback_hours': LAG_MAX_LOOKBACK_HOURS, 'min_pairs': LAG_MIN_PAIRS, 'min_correlation': LAG_MIN_CORRELATION, 'correlation_groups': FEATURE_GROUPS}, ensure_ascii=False, indent=2))


In [ ]:
def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins))})
    for name, matrix in common_features(origins).items():
        frame[name] = matrix.ravel()
    calendar = {'hour': dates.hour, 'minute': dates.minute, 'hour_float': dates.hour + dates.minute / 60.0, 'dayofweek': dates.dayofweek, 'is_weekend': (dates.dayofweek >= 5).astype(np.int8)}
    calendar['hour_sin'] = np.sin(2 * np.pi * calendar['hour_float'] / 24)
    calendar['hour_cos'] = np.cos(2 * np.pi * calendar['hour_float'] / 24)
    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    if h == 6:
        start_minutes = times[origins].hour * 60 + times[origins].minute
        phase = 2 * np.pi * start_minutes / (24 * 60)
        frame['start_sin'] = np.repeat(np.sin(phase), len(routes))
        frame['start_cos'] = np.repeat(np.cos(phase), len(routes))
        daily_maxima, daily_levels, daily_shares = ([], [], [])
        for day in range(1, 8):
            window = np.stack([past(target, slot - day * 48 + offset) for offset in (-1, 0, 1)])
            valid = np.isfinite(window)
            count = valid.sum(axis=0)
            maximum = np.where(valid, window, -np.inf).max(axis=0)
            maximum[count == 0] = np.nan
            level = np.divide(np.where(valid, window, 0).sum(axis=0), count, out=np.full_like(maximum, np.nan), where=count > 0)
            ties = valid & (window == maximum[None, :, :])
            tie_count = ties.sum(axis=0)
            daily_maxima.append(maximum)
            daily_levels.append(level)
            for offset_index, offset in enumerate((-1, 0, 1)):
                share = np.divide(ties[offset_index].astype(np.float32), tie_count, out=np.full_like(maximum, np.nan), where=tie_count > 0)
                daily_shares.append((offset, share))
        frame['peak7_mean_max'] = average_available(daily_maxima).ravel()
        frame['peak7_mean_level'] = average_available(daily_levels).ravel()
        for offset in (-1, 0, 1):
            frame[f'peak7_share_{offset}'] = average_available([share for current_offset, share in daily_shares if current_offset == offset]).ravel()
    frame = add_correlation_features(frame, h, validation_lag_table)
    for days in (1, 2, 7):
        frame[f'slot_days_back_{days}'] = past(target, slot - days * 48).ravel()
    for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
        frame[name] = average_available([past(target, slot - d * 48) for d in days]).ravel()
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day * 48)
        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))
        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()
        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame['overlap'] = np.nan
    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
    return frame.astype({name: 'float32' for name in numeric})

def write_horizon(h):
    stop = len(times) - h
    path = OUT / f'history_h{h}.parquet'
    temporary = path.with_suffix('.partial')
    writer = None
    try:
        for start in range(0, stop, 48):
            origins = np.arange(start, min(start + 48, stop))
            frame = horizon_dataset(h, origins)
            frame['y'] = target[origins + h].ravel()
            table = pa.Table.from_pandas(frame, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(temporary, table.schema, compression='zstd')
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()
    temporary.replace(path)
    features = horizon_dataset(h, np.array([len(times) - 1]))
    requests = test.loc[test.timestamp.eq(end + h * STEP)]
    future = requests.merge(features, on=['route_id', 'timestamp'], how='left', validate='one_to_one')
    if not future.cutoff.eq(end).all() or not future.id.equals(requests.id.reset_index(drop=True)):
        raise ValueError('Ошибка сопоставления тестовых строк')
    future.to_parquet(OUT / f'submission_h{h}.parquet', index=False)
    print(f'h={h}: история {stop * len(routes):,}, тест {len(future):,}')
for h in range(1, 9):
    write_horizon(h)
del values, means, level24, extra_signals, positions


In [ ]:
from catboost import CatBoostRegressor
from catboost.utils import get_gpu_device_count
if get_gpu_device_count() < 1:
    raise RuntimeError('Для обучения включите GPU в настройках Kaggle Notebook.')
MODEL_DIR = OUT.parent / 'models'
MODEL_DIR.mkdir(exist_ok=True)
print('Начало подбора:', tune_start, 'Начало отложенной проверки:', holdout_start)

def make_model(trees):
    return CatBoostRegressor(iterations=trees, depth=7, learning_rate=0.05, loss_function='MAE', eval_metric='MAE', task_type='GPU', devices='0', cat_features=['route_id'], random_seed=42, allow_writing_files=False, verbose=100)

def mean_daily_score(frame, prediction):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(prediction, dtype=np.float64) - actual
    totals = pd.DataFrame({'cutoff': frame.cutoff.dt.normalize().to_numpy(), 'volume': actual, 'error': errors, 'absolute': np.abs(errors)}).groupby('cutoff').sum()
    denominator = totals.volume.where(totals.volume > 0)
    return ((totals.absolute + totals.error.abs()) / denominator).mean()

def score_row(frame, predictions, split):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(predictions, dtype=np.float64) - actual
    volume = actual.sum()
    wape = np.abs(errors).sum() / volume if volume > 0 else np.nan
    signed_bias = errors.sum() / volume if volume > 0 else np.nan
    return {'выборка': split, 'даты': frame.cutoff.dt.normalize().nunique(), 'строки': len(frame), 'WAPE': wape, 'Relative Bias': signed_bias, '|Relative Bias|': abs(signed_bias), 'метрика конкурса': wape + abs(signed_bias), 'среднее по дням': mean_daily_score(frame, predictions)}
TARGET_SCALE = 100000

def model_columns(data, include_pilot_h6=False):
    excluded = ['timestamp', 'cutoff', 'y', 'profile28', 'overlap']
    if not include_pilot_h6:
        excluded.extend((name for name in PILOT_H6_FEATURES if name in data.columns))
    return data.columns.drop(excluded).tolist()
split_dates = {'train': dates[dates < tune_start], 'подбор': dates[(dates >= tune_start) & (dates < holdout_start)], 'test': dates[dates >= holdout_start]}
display(pd.DataFrame([{'выборка': name, 'с': period.min(), 'по': period.max(), 'даты': len(period)} for name, period in split_dates.items()]))
IMPORTANCE_DIR = OUT.parent / 'importance'
IMPORTANCE_DIR.mkdir(exist_ok=True)
builtin_importance_parts = []

def record_builtin_importance(model, columns, h, stage):
    importance = model.get_feature_importance(type='PredictionValuesChange')
    if len(importance) != len(columns):
        raise ValueError('Важности не совпадают с матрицей признаков')
    builtin_importance_parts.append(pd.DataFrame({'feature': columns, 'importance': importance, 'horizon': h, 'stage': stage}))


## Контроль и два варианта очистки

После каждого fit выводятся WAPE, |Relative Bias| и их сумма на одинаковых исходных train/test строках, в том числе на исключённых из обучения целях. Метрика на train также считается отдельно по стартам 10:30. Число деревьев выбирается по tuning MAE. После refit сохраняется прежняя test-оценка, новый test после refit не вычисляется.


In [ ]:
validation_parts = []
train_parts = []
tree_counts = {}
evaluation_tables = {}
for h in range(1, 9):
    data = pd.read_parquet(OUT / f'history_h{h}.parquet')
    columns = model_columns(data)
    train = data.timestamp < tune_start.normalize()
    evaluate = data.cutoff.isin(dates[-28:])
    tune = evaluate & (data.cutoff < holdout_start)
    train_frame = data.loc[train]
    flagged = spike_mask(train_frame, validation_limits)
    meta_columns = ['cutoff', 'timestamp', 'route_id', 'y', 'profile28', 'overlap', 'slot_daily7', 'slot_weekly4']
    evaluation = data.loc[evaluate, meta_columns].copy()
    evaluation['horizon'] = h
    train1030 = train & data.cutoff.isin(dates[:-28])
    training = data.loc[train1030, meta_columns].copy()
    training['horizon'] = h
    for mode, flagged_weight in CLEANING_WEIGHTS.items():
        weights = np.where(flagged, flagged_weight, 1.0)
        model = make_model(1500)
        model.fit(train_frame[columns], train_frame.y / TARGET_SCALE, sample_weight=weights, eval_set=(data.loc[tune, columns], data.loc[tune, 'y'] / TARGET_SCALE), early_stopping_rounds=100, use_best_model=True)
        tree_counts[mode, h] = model.tree_count_
        model.save_model(str(MODEL_DIR / f'{mode}_h{h}_evaluation.cbm'))
        evaluation[mode] = np.maximum(model.predict(data.loc[evaluate, columns]) * TARGET_SCALE, 0)
        train_prediction = np.maximum(model.predict(train_frame[columns]) * TARGET_SCALE, 0)
        training[mode] = train_prediction[train_frame.cutoff.isin(dates[:-28]).to_numpy()]
        test_rows = evaluation.loc[evaluation.cutoff >= holdout_start]
        table = pd.DataFrame([score_row(train_frame, train_prediction, 'train_all_starts'), score_row(training, training[mode], 'train_1030'), score_row(test_rows, test_rows[mode], 'test')])
        table.insert(0, 'horizon', h)
        table.insert(0, 'mode', mode)
        evaluation_tables[mode, h] = table
        print(mode, h, 'trees', model.tree_count_, 'flagged', int(flagged.sum()))
        display(table)
        del model, train_prediction
    validation_parts.append(evaluation)
    train_parts.append(training)
    del data, train_frame
    gc.collect()
validation = pd.concat(validation_parts, ignore_index=True)
training_predictions = pd.concat(train_parts, ignore_index=True)
validation.to_parquet(OUT.parent / 'validation_predictions.parquet', index=False)
training_predictions.to_parquet(OUT.parent / 'train_1030_predictions.parquet', index=False)
pd.concat(evaluation_tables.values(), ignore_index=True).to_csv(OUT.parent / 'model_metrics.csv', index=False)


In [ ]:
def get_raw_full_prediction(frame, w, use_overlap, p_weight):
    if not 0 <= w <= 1 or not 0 <= p_weight <= 1:
        raise ValueError('Веса должны находиться в [0, 1]')
    base = w * frame['catboost'] + (1 - w) * frame['seasonal']
    if use_overlap:
        first = frame.horizon.eq(1)
        base.loc[first] = frame.loc[first, 'overlap']
    later = frame.horizon.gt(1)
    if p_weight > 0:
        base.loc[later] = (1 - p_weight) * base.loc[later] + p_weight * frame.loc[later, 'profile28']
    if not np.isfinite(base.to_numpy(dtype=np.float64)).all():
        raise ValueError('Полный прогноз содержит NaN/inf: проверьте overlap/profile28')
    return base.astype(np.float64)

def apply_final_calibration(raw_pred, calib_type, calib_val):
    if not np.isfinite(calib_val) or not np.isfinite(np.asarray(raw_pred, dtype=np.float64)).all():
        raise ValueError('Неконечное значение прогноза или коэффициента')
    if calib_type == 'none':
        corrected = raw_pred.copy()
    elif calib_type == 'multiplicative':
        if calib_val < 0:
            raise ValueError('Множитель не может быть отрицательным')
        corrected = raw_pred * calib_val
    elif calib_type == 'additive':
        corrected = raw_pred + calib_val
    else:
        raise ValueError(f'Неизвестный тип калибровки: {calib_type}')
    result = corrected.clip(lower=0) if isinstance(corrected, pd.Series) else np.maximum(corrected, 0)
    if not np.isfinite(np.asarray(result, dtype=np.float64)).all():
        raise ValueError('После калибровки получены NaN/inf')
    return result

def fit_calibration_value(frame, raw_pred, calib_type):
    if isinstance(raw_pred, pd.Series) and (not raw_pred.index.equals(frame.index)):
        raise ValueError('Индексы фактов и прогнозов не совпадают')
    actual = frame.y.to_numpy(dtype=np.float64)
    prediction = np.asarray(raw_pred, dtype=np.float64)
    if len(frame) == 0 or prediction.shape != actual.shape:
        raise ValueError('Пустая или несовпадающая calibration-выборка')
    if not np.isfinite(actual).all() or not np.isfinite(prediction).all() or (actual < 0).any():
        raise ValueError('Некорректные факты/прогнозы для калибровки')
    if calib_type == 'none':
        return 0.0
    if calib_type == 'multiplicative':
        total = prediction.sum()
        if not np.isfinite(total) or total <= 0:
            return None
        value = actual.sum() / total
    elif calib_type == 'additive':
        value = np.mean(actual - prediction)
    else:
        raise ValueError(f'Неизвестный тип калибровки: {calib_type}')
    return float(value) if np.isfinite(value) else None

def search_postprocessing(frame):
    tuning = frame.loc[frame.cutoff < holdout_start].copy()
    if tuning.empty:
        raise ValueError('Пустая tuning-выборка')
    best_score, best_config = (np.inf, None)
    rows = []
    for w in np.linspace(0, 1, 11):
        for use_overlap in (False, True):
            for p_weight in np.linspace(0, 1, 11):
                raw = get_raw_full_prediction(tuning, w, use_overlap, p_weight)
                for kind in ('none', 'multiplicative', 'additive'):
                    value = fit_calibration_value(tuning, raw, kind)
                    if value is None:
                        continue
                    score = float(mean_daily_score(tuning, apply_final_calibration(raw, kind, value)))
                    if not np.isfinite(score):
                        continue
                    config = {'w': float(w), 'overlap': bool(use_overlap), 'p_weight': float(p_weight), 'calib_type': kind, 'calib_val': float(value)}
                    rows.append({**config, 'tuning_score': score})
                    if score < best_score:
                        best_score, best_config = (score, config.copy())
    if best_config is None:
        raise ValueError('Не найден конечный кандидат постобработки')
    return (best_score, best_config, pd.DataFrame(rows), tuning)


In [ ]:
def mode_frame(frame, mode):
    result = frame.copy()
    result['catboost'] = result[mode]
    result['seasonal'] = result[['slot_daily7', 'slot_weekly4']].mean(axis=1)
    return result
baseline_frame = mode_frame(validation, 'baseline')
_, config, search, _ = search_postprocessing(baseline_frame)
search.to_csv(OUT.parent / 'baseline_postprocessing_search.csv', index=False)

def predict_processed(frame, mode, calibration):
    prepared = mode_frame(frame, mode)
    raw = get_raw_full_prediction(prepared, config['w'], config['overlap'], config['p_weight'])
    return apply_final_calibration(raw, config['calib_type'], calibration)
tuning_rows = []
for mode in CLEANING_WEIGHTS:
    tuning_frame = validation.loc[validation.cutoff < holdout_start]
    pred = predict_processed(tuning_frame, mode, config['calib_val'])
    tuning_rows.append({'mode': mode, **score_row(tuning_frame, pred, 'tuning')})
tuning_report = pd.DataFrame(tuning_rows)
selected_mode = tuning_report.sort_values('среднее по дням', kind='stable').iloc[0]['mode']
print('Выбор только по tuning:', selected_mode)
display(tuning_report)
tuning_report.to_csv(OUT.parent / 'tuning_selection.csv', index=False)
report_rows = []
for mode in CLEANING_WEIGHTS:
    for split, frame in [('train_1030', training_predictions), ('test', validation.loc[validation.cutoff >= holdout_start])]:
        pred = predict_processed(frame, mode, config['calib_val'])
        report_rows.append({'mode': mode, 'selected_on_tuning': mode == selected_mode, **score_row(frame, pred, split)})
report = pd.DataFrame(report_rows)
display(report)
report.to_csv(OUT.parent / 'independent_metrics_before_refit.csv', index=False)
print('Независимая от выбора по tuning оценка test получена до переобучения. Test уже использовался в предыдущих экспериментах проекта.')
settings = {'selected_mode': selected_mode, 'cleaning_weights': CLEANING_WEIGHTS, 'fixed_baseline_postprocessing': config, 'tune_start': str(tune_start), 'test_start': str(holdout_start), 'test_evaluated_before_refit': True, 'spike_rule': {'median_multiplier': 8, 'mad_multiplier': 12, 'route_median_multiplier': 4, 'min_count': 28}}
(OUT.parent / 'settings.json').write_text(json.dumps(settings, ensure_ascii=False, indent=2))


## Финальное переобучение и сабмит

Выбранный по tuning режим переобучается на всей доступной размеченной истории, включая прежние tuning/test. Если победит исключение всплесков, вся история участвует в построении признаков и статистик, а отмеченные цели получают нулевой вес. Отчёт независимой оценки test выше относится к моделям до этого переобучения. Калибровка фиксируется по tuning, чтобы опыт менял только очистку. Все восемь моделей refit обучаются заново; число деревьев зафиксировано до test.


In [ ]:
full_limits = fit_spike_limits(raw_labels)
full_limits.to_csv(OUT.parent / 'spike_limits_full.csv', index=False)
final_lag_table = calculate_dynamic_shoulder(target, hub_signal, times, routes, fit_before=end + STEP, max_lookback_hours=LAG_MAX_LOOKBACK_HOURS, min_pairs=LAG_MIN_PAIRS, min_correlation=LAG_MIN_CORRELATION)
final_lag_table.to_csv(OUT.parent / 'status6_lags_full.csv', index=False)
submission_parts = []
for h in range(1, 9):
    data = add_correlation_features(pd.read_parquet(OUT / f'history_h{h}.parquet'), h, final_lag_table)
    future = add_correlation_features(pd.read_parquet(OUT / f'submission_h{h}.parquet'), h, final_lag_table)
    columns = model_columns(data)
    flags = spike_mask(data, full_limits)
    weights = np.where(flags, CLEANING_WEIGHTS[selected_mode], 1.0)
    model = make_model(tree_counts[selected_mode, h])
    model.fit(data[columns], data.y / TARGET_SCALE, sample_weight=weights)
    model.save_model(str(MODEL_DIR / f'{selected_mode}_h{h}_full.cbm'))
    result = future[['id', 'profile28', 'overlap', 'slot_daily7', 'slot_weekly4']].copy()
    result['horizon'] = h
    result[selected_mode] = np.maximum(model.predict(future[columns]) * TARGET_SCALE, 0)
    submission_parts.append(result)
    full_prediction = np.maximum(model.predict(data[columns]) * TARGET_SCALE, 0)
    full_row = pd.DataFrame([score_row(data, full_prediction, 'full_history_after_refit')])
    display(pd.concat([evaluation_tables[selected_mode, h], full_row], ignore_index=True))
    del model, data, future, full_prediction
    gc.collect()
predictions = pd.concat(submission_parts, ignore_index=True)
predictions['y_pred'] = predict_processed(predictions, selected_mode, config['calib_val'])
submission = test[['id']].merge(predictions[['id', 'y_pred']], on='id', how='left', validate='one_to_one')
if len(submission) != len(test) or not submission.id.equals(test.id.reset_index(drop=True)) or (not submission.id.is_unique) or (not np.isfinite(submission.y_pred).all()) or (submission.y_pred < 0).any():
    raise ValueError('Некорректный сабмит')
submission.to_csv('submission_stepa_cleaning.csv', index=False)
print('submission_stepa_cleaning.csv', len(submission), 'строк')
display(submission.head())
